In [1]:

from pathlib import Path
import subprocess
import sys

# ============================================================
# CLOUDPULSE AI
# Intelligent Cloud Cost Optimization & Configuration Analyzer
# Local simulation only — no AWS account or paid cloud required.
# ============================================================

APP_CODE = r'''
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(
    page_title="CLOUDPULSE AI",
    page_icon="☁️",
    layout="wide"
)

st.title("☁️ CLOUDPULSE AI")
st.subheader("Intelligent Cloud Cost Optimization & Configuration Risk Analyzer")
st.caption(
    "Local simulation only. No live AWS connection. "
    "All sample costs and resources are fictional."
)

REQUIRED = [
    "resource_id", "resource_name", "resource_type", "region",
    "environment", "owner", "monthly_hours", "cpu_utilization",
    "memory_utilization", "storage_gb", "storage_used_gb",
    "estimated_monthly_cost", "encryption_enabled",
    "public_access_enabled", "iam_scope", "last_activity_days",
    "dependencies"
]

NUMERIC = [
    "monthly_hours", "cpu_utilization", "memory_utilization",
    "storage_gb", "storage_used_gb",
    "estimated_monthly_cost", "last_activity_days"
]


# ------------------------------------------------------------
# 1. SYNTHETIC CLOUD INVENTORY
# ------------------------------------------------------------
def create_sample_data():
    rng = np.random.default_rng(42)
    resource_types = ["EC2", "S3", "RDS", "Lambda"]
    regions = ["ap-south-1", "us-east-1", "eu-west-1"]
    environments = ["development", "staging", "production"]
    owners = ["Data Team", "Engineering", "Analytics", "Research"]
    rows = []

    for i in range(1, 41):
        resource_type = resource_types[(i - 1) % 4]
        storage = int(rng.integers(50, 1500))
        used = int(rng.integers(1, storage + 1))
        cpu = round(float(rng.uniform(2, 95)), 1)
        memory = round(float(rng.uniform(5, 95)), 1)
        hours = int(rng.integers(40, 745))
        cost = round(float(rng.uniform(3, 280)), 2)

        encrypted = bool(rng.choice([True, True, True, False]))
        public = bool(rng.choice([False, False, False, True]))
        iam = str(rng.choice(["least_privilege", "scoped", "broad"]))
        idle_days = int(rng.integers(0, 90))

        # Deliberately inserted examples for demonstrating findings.
        if i in (1, 5, 9):
            cpu, memory, hours, idle_days = 4.0, 12.0, 720, 35
        if i in (2, 6):
            public, encrypted = True, False
        if i in (3, 7):
            iam = "broad"
        if i in (4, 8):
            used = max(1, int(storage * 0.08))

        rows.append({
            "resource_id": f"SIM-{i:03d}",
            "resource_name": f"{resource_type.lower()}-demo-{i:02d}",
            "resource_type": resource_type,
            "region": regions[(i - 1) % 3],
            "environment": environments[(i - 1) % 3],
            "owner": owners[(i - 1) % 4],
            "monthly_hours": hours,
            "cpu_utilization": cpu,
            "memory_utilization": memory,
            "storage_gb": storage,
            "storage_used_gb": min(used, storage),
            "estimated_monthly_cost": cost,
            "encryption_enabled": encrypted,
            "public_access_enabled": public,
            "iam_scope": iam,
            "last_activity_days": idle_days,
            "dependencies": f"SIM-{i-1:03d}" if i > 1 and i % 3 == 0 else ""
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 2. DATA VALIDATION
# ------------------------------------------------------------
def parse_boolean(series):
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False,
        "yes": True, "no": False,
        "y": True, "n": False
    }
    return series.astype(str).str.strip().str.lower().map(mapping)


def validate_data(data):
    missing_columns = [c for c in REQUIRED if c not in data.columns]

    if missing_columns:
        return data, [
            "Missing required columns: " + ", ".join(missing_columns)
        ]

    data = data[REQUIRED].copy()
    errors = []

    for column in NUMERIC:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    for column in ["encryption_enabled", "public_access_enabled"]:
        data[column] = parse_boolean(data[column])

    if data["resource_id"].astype(str).duplicated().any():
        errors.append("Duplicate resource_id values found.")

    if data[NUMERIC].isna().any().any():
        errors.append("Numeric fields contain missing or invalid values.")

    for column in ["cpu_utilization", "memory_utilization"]:
        if ((data[column] < 0) | (data[column] > 100)).any():
            errors.append(f"{column} must be between 0 and 100.")

    nonnegative = [
        "monthly_hours", "storage_gb", "storage_used_gb",
        "estimated_monthly_cost", "last_activity_days"
    ]

    if (data[nonnegative] < 0).any().any():
        errors.append("Hours, storage, cost and inactivity cannot be negative.")

    if (data["storage_used_gb"] > data["storage_gb"]).any():
        errors.append("Storage used cannot exceed total storage.")

    bool_columns = ["encryption_enabled", "public_access_enabled"]
    if data[bool_columns].isna().any().any():
        errors.append("Boolean values must be true/false, yes/no or 1/0.")

    if data["resource_id"].astype(str).str.strip().eq("").any():
        errors.append("Resource IDs cannot be blank.")

    return data, errors


# ------------------------------------------------------------
# 3. RULE-BASED COST AND CONFIGURATION ANALYSIS
# ------------------------------------------------------------
def analyze_resources(data, cpu_limit, memory_limit,
                      unused_limit, idle_limit):

    findings = []

    def add_finding(resource, category, severity, evidence,
                    explanation, recommendation, opportunity=0.0):

        findings.append({
            "finding_id": f"F-{len(findings)+1:04d}",
            "resource_id": resource.resource_id,
            "resource_name": resource.resource_name,
            "category": category,
            "severity": severity,
            "evidence": evidence,
            "explanation": explanation,
            "recommendation": recommendation,
            "estimated_monthly_opportunity": round(
                max(0.0, float(opportunity)), 2
            ),
            "confidence": "Medium — simulated data; verify manually"
        })

    for resource in data.itertuples(index=False):
        cost = float(resource.estimated_monthly_cost)

        # Possible overprovisioning
        if (
            resource.resource_type in ("EC2", "RDS")
            and resource.cpu_utilization < cpu_limit
            and resource.monthly_hours > 300
        ):
            add_finding(
                resource,
                "Possible overprovisioning",
                "Medium",
                f"CPU {resource.cpu_utilization}%; "
                f"{int(resource.monthly_hours)} hours/month",
                "Low CPU utilization with extended runtime may warrant review.",
                "Review peak workload and consider rightsizing or scheduled operation.",
                cost * 0.25
            )

        # Low memory utilization
        if (
            resource.resource_type in ("EC2", "RDS")
            and resource.memory_utilization < memory_limit
            and resource.monthly_hours > 300
        ):
            add_finding(
                resource,
                "Low memory utilization",
                "Low",
                f"Memory {resource.memory_utilization}%",
                "Reported memory use is low, but peaks and application requirements matter.",
                "Review memory trends and performance before resizing.",
                cost * 0.10
            )

        # Underused storage
        if resource.resource_type == "S3" and resource.storage_gb > 0:
            unused_ratio = 1 - (
                resource.storage_used_gb / resource.storage_gb
            )

            if unused_ratio >= unused_limit:
                add_finding(
                    resource,
                    "Low storage utilization",
                    "Low",
                    f"{unused_ratio:.0%} of reported capacity unused",
                    "Inventory suggests storage capacity may exceed current use.",
                    "Check retention, backup, recovery and growth requirements.",
                    cost * min(0.20, unused_ratio * 0.15)
                )

        # Public access indicator
        if resource.public_access_enabled:
            add_finding(
                resource,
                "Public access indicator",
                "High",
                "public_access_enabled=True",
                "Public access may increase exposure depending on policy and data.",
                "Review the access policy and restrict public access if unnecessary."
            )

        # Encryption indicator
        if not resource.encryption_enabled:
            add_finding(
                resource,
                "Encryption indicator",
                "High",
                "encryption_enabled=False",
                "The inventory indicates encryption is not enabled.",
                "Verify requirements and enable suitable encryption if appropriate."
            )

        # Broad IAM permissions
        if str(resource.iam_scope).strip().lower() in {
            "broad", "admin", "wildcard", "*"
        }:
            add_finding(
                resource,
                "Broad IAM permission indicator",
                "High",
                f"iam_scope={resource.iam_scope}",
                "Broad permissions may allow more access than necessary.",
                "Review permissions and apply least-privilege access."
            )

        # Potential inactivity
        if resource.last_activity_days >= idle_limit:
            add_finding(
                resource,
                "Potential inactivity",
                "Medium",
                f"Last activity {int(resource.last_activity_days)} days ago",
                "Inactivity warrants review but does not prove a resource is unused.",
                "Confirm owner, dependencies, schedules and retention before action.",
                cost * 0.15
            )

        # Missing ownership
        if (
            not str(resource.owner).strip()
            or str(resource.owner).lower() == "nan"
        ):
            add_finding(
                resource,
                "Missing owner tag",
                "Low",
                "Owner is blank",
                "Missing ownership makes cost allocation and follow-up harder.",
                "Assign a responsible owner or team."
            )

    columns = [
        "finding_id", "resource_id", "resource_name", "category",
        "severity", "evidence", "explanation", "recommendation",
        "estimated_monthly_opportunity", "confidence"
    ]

    return pd.DataFrame(findings, columns=columns)


# ------------------------------------------------------------
# 4. HELPERS
# ------------------------------------------------------------
def download_csv(data, filename, label):
    st.download_button(
        label=label,
        data=data.to_csv(index=False).encode("utf-8"),
        file_name=filename,
        mime="text/csv",
        use_container_width=True
    )


# ------------------------------------------------------------
# 5. SIDEBAR SETTINGS AND INPUT
# ------------------------------------------------------------
st.sidebar.title("☁️ CLOUDPULSE AI")
st.sidebar.caption("Synthetic data • No live AWS access")
st.sidebar.header("Analysis thresholds")

cpu_limit = st.sidebar.slider("Low CPU threshold (%)", 5, 50, 15)
memory_limit = st.sidebar.slider("Low memory threshold (%)", 5, 50, 20)
unused_limit = st.sidebar.slider("Unused storage ratio", 0.50, 0.99, 0.80, 0.05)
idle_limit = st.sidebar.slider("Inactivity review threshold (days)", 7, 90, 30)

uploaded_file = st.sidebar.file_uploader(
    "Upload cloud inventory CSV",
    type=["csv"]
)

if uploaded_file is not None:
    try:
        raw_data = pd.read_csv(uploaded_file)
        data_source = f"Uploaded file: {uploaded_file.name}"
    except Exception as error:
        st.error(f"Unable to read CSV: {error}")
        st.stop()
else:
    raw_data = create_sample_data()
    data_source = "Built-in synthetic sample dataset"

df, validation_errors = validate_data(raw_data)

if validation_errors:
    st.error("Please correct the inventory data before analysis.")
    for error in validation_errors:
        st.write("• " + error)
    st.info("Required columns:")
    st.code(", ".join(REQUIRED))
    st.stop()

findings = analyze_resources(
    df, cpu_limit, memory_limit, unused_limit, idle_limit
)

if findings.empty:
    recommendations = pd.DataFrame(
        columns=["priority", "resource_id", "recommendation", "reason"]
    )
    opportunity = 0.0
else:
    severity_rank = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
    rec = findings.copy()
    rec["rank"] = rec["severity"].map(severity_rank).fillna(9)
    rec["priority"] = rec["severity"]
    rec["reason"] = rec["evidence"] + " — " + rec["explanation"]

    recommendations = (
        rec.sort_values(
            ["rank", "estimated_monthly_opportunity"],
            ascending=[True, False]
        )[
            ["priority", "resource_id", "recommendation", "reason"]
        ].drop_duplicates()
    )

    # Avoid summing several estimated opportunities for one resource.
    opportunity = float(
        findings.groupby("resource_id")[
            "estimated_monthly_opportunity"
        ].max().sum()
    )

total_cost = float(df["estimated_monthly_cost"].sum())

pages = [
    "Overview", "Resource Inventory", "Waste & Risk Findings",
    "Cost Analysis", "What-If Simulator", "Dependencies",
    "Recommendations", "Data Quality", "About"
]

page = st.sidebar.radio("Navigate", pages)

st.info(
    f"**Data source:** {data_source}  |  "
    f"**Generated:** {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}"
)


# ------------------------------------------------------------
# 6. OVERVIEW DASHBOARD
# ------------------------------------------------------------
if page == "Overview":
    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Simulated resources", f"{len(df):,}")
    col2.metric("Estimated monthly spend", f"${total_cost:,.2f}")
    col3.metric("Potential monthly opportunity", f"${opportunity:,.2f}")
    col4.metric(
        "High-priority findings",
        int(findings["severity"].isin(["Critical", "High"]).sum())
        if not findings.empty else 0
    )

    st.caption(
        "Potential savings are illustrative, not guaranteed. "
        "They are not official AWS pricing estimates."
    )

    left, right = st.columns(2)

    with left:
        by_type = (
            df.groupby("resource_type", as_index=False)
            ["estimated_monthly_cost"].sum()
        )
        st.plotly_chart(
            px.bar(
                by_type,
                x="resource_type",
                y="estimated_monthly_cost",
                title="Estimated spend by resource type",
                labels={
                    "resource_type": "Resource type",
                    "estimated_monthly_cost": "Simulated USD/month"
                }
            ),
            use_container_width=True
        )

    with right:
        if findings.empty:
            st.success("No findings for the selected thresholds.")
        else:
            severity_counts = (
                findings.groupby("severity", as_index=False).size()
            )
            st.plotly_chart(
                px.pie(
                    severity_counts,
                    names="severity",
                    values="size",
                    hole=0.4,
                    title="Findings by severity"
                ),
                use_container_width=True
            )

    st.subheader("Latest findings")
    st.dataframe(findings.head(10), use_container_width=True, hide_index=True)
    st.warning(
        "These are simulated review indicators, not confirmed vulnerabilities."
    )


# ------------------------------------------------------------
# 7. RESOURCE INVENTORY
# ------------------------------------------------------------
elif page == "Resource Inventory":
    st.subheader("Resource inventory explorer")

    col1, col2, col3 = st.columns(3)

    selected_types = col1.multiselect(
        "Resource type",
        sorted(df["resource_type"].unique()),
        default=sorted(df["resource_type"].unique())
    )

    selected_regions = col2.multiselect(
        "Region",
        sorted(df["region"].unique()),
        default=sorted(df["region"].unique())
    )

    selected_envs = col3.multiselect(
        "Environment",
        sorted(df["environment"].unique()),
        default=sorted(df["environment"].unique())
    )

    view = df[
        df["resource_type"].isin(selected_types)
        & df["region"].isin(selected_regions)
        & df["environment"].isin(selected_envs)
    ]

    st.write(f"Showing **{len(view)}** resources.")
    st.dataframe(view, use_container_width=True, hide_index=True)

    download_csv(
        view, "cloudpulse_filtered_inventory.csv",
        "Download filtered inventory"
    )


# ------------------------------------------------------------
# 8. FINDINGS
# ------------------------------------------------------------
elif page == "Waste & Risk Findings":
    st.subheader("Waste and configuration risk findings")

    if findings.empty:
        st.success("No findings under the current thresholds.")
    else:
        col1, col2 = st.columns(2)

        selected_severities = col1.multiselect(
            "Severity",
            ["Critical", "High", "Medium", "Low"],
            default=["Critical", "High", "Medium", "Low"]
        )

        selected_categories = col2.multiselect(
            "Category",
            sorted(findings["category"].unique()),
            default=sorted(findings["category"].unique())
        )

        view = findings[
            findings["severity"].isin(selected_severities)
            & findings["category"].isin(selected_categories)
        ]

        st.dataframe(view, use_container_width=True, hide_index=True)

        download_csv(
            view, "cloudpulse_findings.csv",
            "Download findings CSV"
        )

    st.caption(
        "Severity indicates review priority. It does not prove an actual "
        "security vulnerability."
    )


# ------------------------------------------------------------
# 9. COST ANALYSIS
# ------------------------------------------------------------
elif page == "Cost Analysis":
    st.subheader("Cost intelligence")

    left, right = st.columns(2)

    with left:
        by_owner = (
            df.groupby("owner", as_index=False)
            ["estimated_monthly_cost"].sum()
            .sort_values("estimated_monthly_cost", ascending=False)
        )
        st.plotly_chart(
            px.bar(
                by_owner,
                x="owner",
                y="estimated_monthly_cost",
                title="Estimated spend by owner",
                labels={"estimated_monthly_cost": "Simulated USD/month"}
            ),
            use_container_width=True
        )

    with right:
        by_region = (
            df.groupby("region", as_index=False)
            ["estimated_monthly_cost"].sum()
            .sort_values("estimated_monthly_cost", ascending=False)
        )
        st.plotly_chart(
            px.bar(
                by_region,
                x="region",
                y="estimated_monthly_cost",
                title="Estimated spend by region",
                labels={"estimated_monthly_cost": "Simulated USD/month"}
            ),
            use_container_width=True
        )

    st.subheader("Highest estimated-cost resources")
    st.dataframe(
        df.sort_values("estimated_monthly_cost", ascending=False)[[
            "resource_id", "resource_name", "resource_type",
            "owner", "region", "estimated_monthly_cost"
        ]],
        use_container_width=True,
        hide_index=True
    )

    st.caption(
        "Total estimated monthly spend = sum of the inventory's "
        "estimated_monthly_cost field. Values are fictional."
    )

    download_csv(by_owner, "cloudpulse_cost_by_owner.csv", "Download cost summary")


# ------------------------------------------------------------
# 10. WHAT-IF SIMULATOR
# ------------------------------------------------------------
elif page == "What-If Simulator":
    st.subheader("What-if savings simulator")
    st.write(
        "Explore a hypothetical cost reduction for review candidates. "
        "No cloud resource is modified."
    )

    reduction = st.slider(
        "Hypothetical reduction (%)", 0, 50, 15, 5
    )
    minimum_cost = st.slider(
        "Minimum monthly cost to include (USD)", 0, 250, 20, 5
    )

    candidates = df[
        (df["estimated_monthly_cost"] >= minimum_cost)
        & (
            (df["cpu_utilization"] < cpu_limit)
            | (df["last_activity_days"] >= idle_limit)
        )
    ]

    candidate_spend = float(candidates["estimated_monthly_cost"].sum())
    projected_savings = candidate_spend * reduction / 100

    col1, col2, col3 = st.columns(3)
    col1.metric("Review candidates", len(candidates))
    col2.metric("Candidate spend/month", f"${candidate_spend:,.2f}")
    col3.metric("Hypothetical savings/month", f"${projected_savings:,.2f}")

    st.dataframe(
        candidates[[
            "resource_id", "resource_name", "resource_type",
            "cpu_utilization", "last_activity_days",
            "estimated_monthly_cost"
        ]],
        use_container_width=True,
        hide_index=True
    )

    st.caption(
        "This is a scenario, not a forecast. Validate workload peaks, "
        "contracts, dependencies and actual pricing before taking action."
    )


# ------------------------------------------------------------
# 11. DEPENDENCY EXPLORER
# ------------------------------------------------------------
elif page == "Dependencies":
    st.subheader("Resource dependency explorer")

    known_ids = set(df["resource_id"].astype(str))
    edges = []

    for resource in df.itertuples(index=False):
        dependency_text = str(resource.dependencies).strip()

        if dependency_text and dependency_text.lower() != "nan":
            for target in [
                value.strip()
                for value in dependency_text.split(";")
                if value.strip()
            ]:
                if target in known_ids:
                    edges.append({
                        "source_resource": resource.resource_id,
                        "depends_on": target
                    })

    if edges:
        edge_df = pd.DataFrame(edges)
        st.dataframe(edge_df, use_container_width=True, hide_index=True)
        download_csv(
            edge_df, "cloudpulse_dependencies.csv",
            "Download dependency list"
        )
    else:
        st.info("No valid dependency links were found.")

    st.caption(
        "Dependencies are taken from the inventory and do not prove "
        "real runtime connectivity or blast radius."
    )


# ------------------------------------------------------------
# 12. PRIORITIZED RECOMMENDATIONS
# ------------------------------------------------------------
elif page == "Recommendations":
    st.subheader("Prioritized recommendations")

    if recommendations.empty:
        st.success("No recommendations available for these thresholds.")
    else:
        st.dataframe(
            recommendations,
            use_container_width=True,
            hide_index=True
        )
        download_csv(
            recommendations, "cloudpulse_recommendations.csv",
            "Download recommendations CSV"
        )

    st.markdown("### Human review checklist")

    checklist = [
        "Confirm resource owner and business purpose",
        "Review usage trends and peak demand",
        "Check dependencies, backups, retention and recovery",
        "Verify actual pricing and savings assumptions",
        "Obtain approval before making real changes"
    ]

    for item in checklist:
        st.checkbox(item, key="check_" + str(checklist.index(item)))


# ------------------------------------------------------------
# 13. DATA QUALITY REPORT
# ------------------------------------------------------------
elif page == "Data Quality":
    st.subheader("Data quality report")

    quality = pd.DataFrame({
        "check": [
            "Rows", "Columns", "Duplicate resource IDs",
            "Missing values", "Invalid CPU values",
            "Invalid memory values", "Storage used exceeds capacity"
        ],
        "result": [
            len(df),
            len(df.columns),
            int(df["resource_id"].duplicated().sum()),
            int(df.isna().sum().sum()),
            int(((df["cpu_utilization"] < 0) |
                 (df["cpu_utilization"] > 100)).sum()),
            int(((df["memory_utilization"] < 0) |
                 (df["memory_utilization"] > 100)).sum()),
            int((df["storage_used_gb"] > df["storage_gb"]).sum())
        ]
    })

    st.dataframe(quality, use_container_width=True, hide_index=True)

    st.markdown("### Required CSV columns")
    st.code(", ".join(REQUIRED))

    st.markdown("### Dataset preview")
    st.dataframe(df.head(10), use_container_width=True, hide_index=True)

    download_csv(
        quality, "cloudpulse_data_quality.csv",
        "Download data quality report"
    )


# ------------------------------------------------------------
# 14. ABOUT AND LIMITATIONS
# ------------------------------------------------------------
else:
    st.subheader("About CLOUDPULSE AI")

    st.markdown("""
    **Purpose**

    CLOUDPULSE AI is a local cloud-inventory analysis simulator. It
    demonstrates data ingestion, validation, cost aggregation, configuration
    review indicators, visual analytics and human-reviewed recommendations.

    **Technology stack**

    - Python
    - Pandas and NumPy
    - Streamlit
    - Plotly

    **Current limitations**

    - The built-in inventory is synthetic.
    - It does not connect to AWS or another cloud provider.
    - It does not verify real configurations or perform remediation.
    - Costs and potential savings are illustrative.
    - Rules can produce false positives and require human review.

    **Possible future extensions**

    - Add unit tests and a model evaluation module.
    - Add a read-only cloud inventory importer after security review.
    - Add role-based access, audit logs and authenticated users.
    - Add a separately reviewed cloud integration only when needed.
    """)

    st.code(
        "pip install streamlit pandas numpy plotly\n"
        "streamlit run cloudpulse_ai.py",
        language="bash"
    )

st.divider()
st.caption(
    "CLOUDPULSE AI • Educational portfolio project • "
    "Synthetic data only • No live cloud actions"
)
'''

# Write the full application to a Python file.
project_dir = Path("CLOUDPULSE_AI")
project_dir.mkdir(exist_ok=True)

app_file = project_dir / "cloudpulse_ai.py"
app_file.write_text(APP_CODE, encoding="utf-8")

requirements_file = project_dir / "requirements.txt"
requirements_file.write_text(
    "streamlit\npandas\nnumpy\nplotly\n",
    encoding="utf-8"
)

print(" CLOUDPULSE AI application created successfully!")
print("Application:", app_file.resolve())
print("Requirements:", requirements_file.resolve())
print()
print("NEXT STEPS:")
print("1. Install dependencies: pip install -r CLOUDPULSE_AI/requirements.txt")
print("2. Open a terminal in the same environment.")
print("3. Run: streamlit run CLOUDPULSE_AI/cloudpulse_ai.py")
print()
print("Note: This is a local simulator. It does not connect to AWS.")


 CLOUDPULSE AI application created successfully!
Application: /content/CLOUDPULSE_AI/cloudpulse_ai.py
Requirements: /content/CLOUDPULSE_AI/requirements.txt

NEXT STEPS:
1. Install dependencies: pip install -r CLOUDPULSE_AI/requirements.txt
2. Open a terminal in the same environment.
3. Run: streamlit run CLOUDPULSE_AI/cloudpulse_ai.py

Note: This is a local simulator. It does not connect to AWS.
